# Quality Investing in Nordic Equity Markets
### A Bayesian Approach to Accounting Signal Precision

This notebook presents the full empirical analysis for the thesis. It is organised
into seven sections:

1. **Descriptive Statistics** — Sample composition, sector and exchange distribution
2. **Model Diagnostics** — Shrinkage behaviour, uncertainty estimates, hyperparameters
3. **Method Comparison** — Rank correlations across the four sorting signals
4. **Return Analysis** — Portfolio returns under flexible sorting and filtering
5. **Rolling Sharpe** — Time-varying risk-adjusted performance
6. **Uncertainty Portfolios** — Returns sorted by accounting noise (σ)
7. **Significance Tests** — Factor-model alphas across methods and factor models

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
from scipy import stats
import statsmodels.api as sm
import sys
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ── Plot style ────────────────────────────────────────────────────────────────
NOTEBOOK_DIR_STYLE = Path.cwd()
ROOT_STYLE = NOTEBOOK_DIR_STYLE.parent if NOTEBOOK_DIR_STYLE.name == 'res_visualisation' else NOTEBOOK_DIR_STYLE

import sys
if str(ROOT_STYLE) not in sys.path:
    sys.path.insert(0, str(ROOT_STYLE))

from plot_style_utils import apply_plot_theme, PALETTE, THEME, style_axis

apply_plot_theme()

print("Plot theme applied.")
print(f"Available palette colours: {list(PALETTE.keys())}")

In [ ]:
# ── Run configuration ─────────────────────────────────────────────────────────
# ════════════════════════════════════════════════════════════════
#  CHANGE THIS WHEN SWITCHING RUN
RUN = 'current_res'
# ════════════════════════════════════════════════════════════════

START_YEAR = 2010

METHODS = ["Method1_Raw", "Method2_PostMean", "Method3_ProbQ5"]

LABELS = {
    "Method1_Raw":        "Raw signal",
    "Method2_PostMean":   "Posterior mean",
    "Method3_ProbQ5":     "P(top quintile)",
}

SIGNAL_COL = {
    "Method1_Raw":          "theta_obs",
    "Method2_PostMean":     "theta_post_mean",
    "Method3_ProbQ5":       "p_q5",
}

METHOD_COLORS = {
    "Method1_Raw":        PALETTE["blue"],
    "Method2_PostMean":   PALETTE["orange"],
    "Method3_ProbQ5":     PALETTE["green"],
    #"Method4_ProbMedian": PALETTE["teal"],
}

QUINTILES = ["Q1", "Q2", "Q3", "Q4", "Q5"]

FACTOR_MODELS = {
    "CAPM":    ["MKT"],
    "FF3":     ["MKT", "SMB", "HML"],
    "FF3+MOM": ["MKT", "SMB", "HML", "MOM"],
    "FF5":     ["MKT", "SMB", "HML", "RMW", "CMA"],
    "FF5+MOM": ["MKT", "SMB", "HML", "RMW", "CMA", "MOM"],
}

INTERNAL_FACTOR_MODELS = {
    "CAPM": "CAPM",
    "FF3": "FF3",
    "FF3+MOM": "Carhart",
    "FF5": "FF5",
    "FF5+MOM": "FF5_MOM",
}

print(f"Run: {RUN}")
print(f"Methods: {list(LABELS.values())}")


In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'res_visualisation' else NOTEBOOK_DIR

RUN_DIR       = ROOT / 'results' / RUN
LATENT_DIR    = RUN_DIR / 'latent_prof_model' / 'HB'
PORT_FORM_DIR = RUN_DIR / 'portfolio_formation' / 'HB'
PORT_EVAL_DIR = RUN_DIR / 'portfolio_evaluation' / 'HB'
STEP2_CSV     = ROOT / 'results' / 'extraction_static' / 'prepared_step2_input.csv'
FACTORS_CSV   = ROOT / 'results' / 'extraction_static' / 'factor_data.csv'
PRICES_CSV    = ROOT / 'data/processed_data_lseg/all_stock_prices_nok.csv'
MKTCAP_CSV    = ROOT / 'data/processed_data_lseg/historical_market_cap_nok.csv'
MONTHLY_HOLDINGS_CSV = PORT_EVAL_DIR / 'monthly_holdings.csv'

SHARED_DIR = ROOT / 'modelling' / 'shared'
if str(SHARED_DIR) not in sys.path:
    sys.path.insert(0, str(SHARED_DIR))

from step5_evaluation import alpha_differences as stacked_alpha_differences

# ── Validate ──────────────────────────────────────────────────────────────────
required = {
    'Latent HB':         LATENT_DIR,
    'Portfolio form':    PORT_FORM_DIR,
    'Portfolio eval HB': PORT_EVAL_DIR,
    'Monthly holdings':  MONTHLY_HOLDINGS_CSV,
    'Step2 input':       STEP2_CSV,
    'Factor data':       FACTORS_CSV,
    'Prices':            PRICES_CSV,
    'Market cap':        MKTCAP_CSV,
}

all_ok = True
for label, path in required.items():
    ok = path.exists()
    all_ok = all_ok and ok
    print(f"  {'✓' if ok else '✗ NOT FOUND':12s} {label:20s} {path}")

if not all_ok:
    raise FileNotFoundError("Some paths are missing — check the RUN name above.")

# ── Load data ─────────────────────────────────────────────────────────────────
firm_year = pd.read_csv(LATENT_DIR / 'latent_prof_firm_year.csv', low_memory=False)
port_long = pd.read_csv(PORT_FORM_DIR / 'portfolio_assignments_long.csv')
factors   = pd.read_csv(FACTORS_CSV, parse_dates=['Date']).set_index('Date')
step2     = pd.read_csv(STEP2_CSV)

# ── Exchange labels ───────────────────────────────────────────────────────────
EXCHANGE_MAP = {
    "OL": "Oslo (OL)", "ST": "Stockholm (ST)",
    "HE": "Helsinki (HE)", "CO": "Copenhagen (CO)", "IC": "Iceland (IC)"
}

firm_year['Exchange'] = firm_year['Ticker'].str.split('.').str[-1].map(EXCHANGE_MAP).fillna('Other')

# ── EK ratio ──────────────────────────────────────────────────────────────────
step2[['BE', 'AT']] = step2[['BE', 'AT']].apply(pd.to_numeric, errors='coerce')
equity_ratio = step2[
    step2['BE'].notna() & step2['AT'].notna() &
    (step2['AT'] > 0) & (step2['BE'] > 0)
].copy()
equity_ratio['EK_ratio']      = equity_ratio['BE'] / equity_ratio['AT']
equity_ratio['FormationYear'] = equity_ratio['Year'] + 1

print(f"\nLoaded successfully:")
print(f"  firm_year:  {firm_year.shape}")
print(f"  port_long:  {port_long.shape}")
print(f"  factors:    {factors.shape}")
print(f"  step2:      {step2.shape}")


In [ ]:
def load_holdings_from_saved(path):
    """
    Loads the repo's portfolio_evaluation/monthly_holdings.csv output.
    This is the canonical monthly constituent file used for portfolio returns.
    """
    holdings = pd.read_csv(path, parse_dates=['Date'], low_memory=False)

    required_cols = ['Ticker', 'Date', 'Return', 'LagMarketCap', 'FormationYear', 'Method', 'Portfolio']
    missing = [c for c in required_cols if c not in holdings.columns]
    if missing:
        raise ValueError(f"monthly_holdings.csv is missing required columns: {missing}")

    for col in ['Return', 'LagMarketCap', 'FormationYear']:
        holdings[col] = pd.to_numeric(holdings[col], errors='coerce')

    holdings = holdings[
        holdings['Return'].notna() &
        holdings['LagMarketCap'].notna() &
        (holdings['LagMarketCap'] > 0) &
        holdings['FormationYear'].notna()
    ].copy()
    holdings['FormationYear'] = holdings['FormationYear'].astype(int)
    return holdings.reset_index(drop=True)


def build_holdings_from_raw(prices_wide, mktcap_wide, port_long):
    """
    Fallback only: rebuild holdings from raw prices, market caps, and assignments.
    Uses the same calendar-year FormationYear convention as modelling/shared/helper_functions.py.
    """

    prices = prices_wide.melt(id_vars='Ticker', var_name='Date', value_name='Price')
    prices['Date'] = pd.to_datetime(prices['Date'])
    prices = prices.sort_values(['Ticker', 'Date']).reset_index(drop=True)
    prices['Return'] = prices.groupby('Ticker')['Price'].pct_change(fill_method=None)
    prices = prices.drop(columns='Price')

    mktcap = mktcap_wide.melt(id_vars='Ticker', var_name='Date', value_name='MarketCap')
    mktcap['Date'] = pd.to_datetime(mktcap['Date'])
    mktcap = mktcap.sort_values(['Ticker', 'Date']).reset_index(drop=True)
    mktcap['LagMarketCap'] = mktcap.groupby('Ticker')['MarketCap'].shift(1)
    mktcap = mktcap.drop(columns='MarketCap')

    monthly = prices.merge(mktcap, on=['Ticker', 'Date'], how='inner')
    monthly['FormationYear'] = monthly['Date'].dt.year

    holdings = monthly.merge(
        port_long[['Ticker', 'FormationYear', 'Method', 'Portfolio']],
        on=['Ticker', 'FormationYear'],
        how='inner'
    )

    holdings = holdings[
        holdings['Return'].notna() &
        holdings['LagMarketCap'].notna() &
        (holdings['LagMarketCap'] > 0)
    ].reset_index(drop=True)

    return holdings


# ── Load canonical monthly holdings ────────────────────────────────────────────
import time
t0 = time.perf_counter()

if MONTHLY_HOLDINGS_CSV.exists():
    holdings_all = load_holdings_from_saved(MONTHLY_HOLDINGS_CSV)
    holdings_source = MONTHLY_HOLDINGS_CSV
else:
    print("monthly_holdings.csv not found; rebuilding holdings from raw files...")
    prices_wide = pd.read_csv(PRICES_CSV)
    mktcap_wide = pd.read_csv(MKTCAP_CSV)
    holdings_all = build_holdings_from_raw(prices_wide, mktcap_wide, port_long)
    holdings_source = 'rebuilt from raw prices and market caps'

holdings_all['Exchange'] = (
    holdings_all['Ticker']
    .astype(str)
    .str.split('.')
    .str[-1]
    .map(EXCHANGE_MAP)
    .fillna('Other')
)

print(f"Loaded holdings in {time.perf_counter() - t0:.1f}s")
print(f"Holdings source: {holdings_source}")
print(f"Holdings shape:  {holdings_all.shape}")
print(f"Date range:      {holdings_all['Date'].min().date()} → {holdings_all['Date'].max().date()}")
print(f"Methods:         {holdings_all['Method'].unique().tolist()}")
print(f"Portfolios:      {sorted(holdings_all['Portfolio'].dropna().unique().tolist())}")
print(f"Exchanges:       {sorted(holdings_all['Exchange'].unique().tolist())}")

sample = holdings_all[holdings_all['FormationYear'] == 2012]
print(f"\nVerification — FormationYear=2012:")
print(f"  Date range: {sample['Date'].min().date()} → {sample['Date'].max().date()}")
print(f"  Expected:   2012-01-31 → 2012-12-31")


In [ ]:
PORTFOLIO_CHOICES = [
    # Single quintiles
    ("Q1",           ["Q1"]),
    ("Q2",           ["Q2"]),
    ("Q3",           ["Q3"]),
    ("Q4",           ["Q4"]),
    ("Q5",           ["Q5"]),
    # Bottom / top 40%
    ("Q1+Q2",        ["Q1", "Q2"]),
    ("Q4+Q5",        ["Q4", "Q5"]),
    # 60/40 split
    ("Q1+Q2+Q3",     ["Q1", "Q2", "Q3"]),
    ("Q2+Q3+Q4",     ["Q2", "Q3", "Q4"]),
    ("Q3+Q4+Q5",     ["Q3", "Q4", "Q5"]),
]

SHORT_CHOICES = [("None", [])] + PORTFOLIO_CHOICES

PORTFOLIO_LOOKUP = {label: quintiles for label, quintiles in PORTFOLIO_CHOICES}
SHORT_LOOKUP     = {label: quintiles for label, quintiles in SHORT_CHOICES}


# ── EK filter helper ──────────────────────────────────────────────────────────
EK_FILTERS = {
    "No filter":   0.00,
    "EK/TK > 5%":  0.05,
    "EK/TK > 10%": 0.10,
    "EK/TK > 20%": 0.20,
}

def apply_ek_filter(holdings_df, threshold):
    if threshold == 0.0:
        return holdings_df
    valid = equity_ratio[equity_ratio['EK_ratio'] >= threshold][
        ['Ticker', 'FormationYear']
    ].drop_duplicates()
    return holdings_df.merge(valid, on=['Ticker', 'FormationYear'], how='inner')


# ── Size filter helper ────────────────────────────────────────────────────────
def apply_size_filter(holdings_df, size_filter):
    """
    'All'        → no filter
    'Large Cap'  → top 50% by median LagMarketCap per FormationYear
    'Small Cap'  → bottom 50% by median LagMarketCap per FormationYear
    """
    if size_filter == "All":
        return holdings_df

    median_mktcap = (
        holdings_df
        .groupby(['Ticker', 'FormationYear'])['LagMarketCap']
        .median()
        .reset_index()
        .rename(columns={'LagMarketCap': 'MedianMktCap'})
    )

    yearly_median = (
        median_mktcap
        .groupby('FormationYear')['MedianMktCap']
        .median()
        .reset_index()
        .rename(columns={'MedianMktCap': 'YearlyMedian'})
    )

    median_mktcap = median_mktcap.merge(yearly_median, on='FormationYear')

    if size_filter == "Large Cap":
        valid = median_mktcap[median_mktcap['MedianMktCap'] >= median_mktcap['YearlyMedian']]
    else:  # Small Cap
        valid = median_mktcap[median_mktcap['MedianMktCap'] < median_mktcap['YearlyMedian']]

    return holdings_df.merge(
        valid[['Ticker', 'FormationYear']], on=['Ticker', 'FormationYear'], how='inner'
    )


# ── Exchange filter helper ────────────────────────────────────────────────────
def apply_exchange_filter(holdings_df, exchange):
    if exchange == "All":
        return holdings_df
    return holdings_df[holdings_df['Exchange'] == exchange].copy()


# ── Industry neutralisation helper ───────────────────────────────────────────
def assign_industry_neutral_portfolios(holdings_df, signal_col):
    """
    Re-ranks firms into quintiles within each sector × FormationYear × Method.
    Requires signal_col to be present in holdings_df.
    """
    pieces = []
    for (fy, sector, method), sub in holdings_df.groupby(
        ['FormationYear', 'Sector', 'Method']
    ):
        unique_firms = sub.drop_duplicates('Ticker').copy()
        if len(unique_firms) < 5:
            continue
        unique_firms['Portfolio'] = 'Q' + pd.qcut(
            unique_firms[signal_col].rank(method='first'),
            q=5, labels=['1', '2', '3', '4', '5']
        ).astype(str)
        pieces.append(unique_firms[['Ticker', 'FormationYear', 'Method', 'Portfolio']])

    if not pieces:
        return pd.DataFrame(columns=['Ticker', 'FormationYear', 'Method', 'Portfolio'])
    return pd.concat(pieces, ignore_index=True)


# ── Core return computation ───────────────────────────────────────────────────
def compute_monthly_returns(holdings_df, weighting, long_quintiles, short_quintiles):
    """
    Computes monthly long, short, and L/S spread returns.

    Parameters
    ----------
    holdings_df     : filtered holdings DataFrame
    weighting       : 'VW' or 'EW'
    long_quintiles  : list of quintile labels e.g. ['Q4', 'Q5']
    short_quintiles : list of quintile labels e.g. ['Q1', 'Q2'], or []

    Returns
    -------
    dict with keys 'long', 'short' (or None), 'spread' (or None)
    each value is a DataFrame with columns: Date, Method, Return
    """

    def _portfolio_return(df, quintiles):
        sub = df[df['Portfolio'].isin(quintiles)]
        if weighting == 'VW':
            monthly = (
                sub.groupby(['Date', 'Method'])
                .apply(lambda g: (g['Return'] * g['LagMarketCap']).sum() / g['LagMarketCap'].sum())
                .reset_index(name='Return')
            )
        else:
            monthly = (
                sub.groupby(['Date', 'Method'])['Return']
                .mean()
                .reset_index()
            )
        return monthly.sort_values(['Method', 'Date']).reset_index(drop=True)

    long_ret  = _portfolio_return(holdings_df, long_quintiles)
    short_ret = _portfolio_return(holdings_df, short_quintiles) if short_quintiles else None

    if short_ret is not None:
        spread = long_ret.merge(
            short_ret, on=['Date', 'Method'], suffixes=('_long', '_short')
        )
        spread['Return'] = spread['Return_long'] - spread['Return_short']
        spread = spread[['Date', 'Method', 'Return']]
    else:
        spread = None

    return {'long': long_ret, 'short': short_ret, 'spread': spread}


print("Portfolio choices and return computation functions ready.")
print(f"Long/short options: {[label for label, _ in PORTFOLIO_CHOICES]}")



In [ ]:
# ── Pre-computation of the 48 filtered holdings DataFrames ───────────────────
import time

EXCHANGES    = ["All"] + sorted(holdings_all['Exchange'].dropna().unique().tolist())
SIZE_FILTERS = ["All", "Large Cap", "Small Cap"]

# Add sector info to holdings_all if not already present
if 'Sector' not in holdings_all.columns:
    sector_map = (
        step2[['Ticker', 'Year', 'Sector']]
        .rename(columns={'Year': 'FormationYear'})
        .assign(FormationYear=lambda x: x['FormationYear'] + 1)
        .drop_duplicates()
    )
    holdings_all = holdings_all.merge(sector_map, on=['Ticker', 'FormationYear'], how='left')
    print(f"Sector coverage: {holdings_all['Sector'].notna().mean():.1%}")

# Add signal columns from firm_year for industry neutralisation
signal_cols = list(SIGNAL_COL.values())
missing_signals = [c for c in signal_cols if c not in holdings_all.columns]
if missing_signals:
    firm_year_signals = firm_year[['Ticker', 'FormationYear'] + signal_cols].drop_duplicates()
    holdings_all = holdings_all.merge(firm_year_signals, on=['Ticker', 'FormationYear'], how='left')
    print(f"Signal columns added: {signal_cols}")

# ── Pre-compute ───────────────────────────────────────────────────────────────
PRECOMPUTED = {}

combinations = [
    (w, n, s, ek_label)
    for w        in ["VW", "EW"]
    for n        in [False, True]
    for s        in SIZE_FILTERS
    for ek_label in EK_FILTERS.keys()
]

print(f"\nPre-computing {len(combinations)} combinations...")
t0_total = time.perf_counter()

for i, (w, n, s, ek_label) in enumerate(combinations):
    t0 = time.perf_counter()

    # 1. Apply EK filter
    df = apply_ek_filter(holdings_all, EK_FILTERS[ek_label])

    # 2. Apply size filter
    df = apply_size_filter(df, s)

    # 3. Apply industry neutralisation (re-assign portfolios within sectors)
    if n:
        new_portfolios = []
        for method, signal_col in SIGNAL_COL.items():
            if signal_col not in df.columns:
                continue
            method_df    = df[df['Method'] == method].copy()
            new_assigned = assign_industry_neutral_portfolios(method_df, signal_col)
            new_portfolios.append(new_assigned)

        if new_portfolios:
            new_assign = pd.concat(new_portfolios, ignore_index=True)
            df = df.drop(columns='Portfolio').merge(
                new_assign, on=['Ticker', 'FormationYear', 'Method'], how='inner'
            )

    PRECOMPUTED[(w, n, s, ek_label)] = df

    elapsed = time.perf_counter() - t0
    print(f"  [{i+1:2d}/{len(combinations)}] {w} | {'Industry neutral' if n else 'Standard':17s} | {s:10s} | {ek_label:12s} → {len(df):,} rows  ({elapsed:.1f}s)")

total = time.perf_counter() - t0_total
print(f"\nDone. Total pre-computation time: {total:.0f}s ({total/60:.1f} min)")
print(f"Stored {len(PRECOMPUTED)} combinations in PRECOMPUTED dict.")

## 1. Descriptive Statistics

This section provides an overview of the sample composition, including the distribution
of firms across formation years, exchanges, and sectors. All statistics are based on
the HB latent quality model output.

In [ ]:
# ── Sample summary ────────────────────────────────────────────────────────────
df = firm_year.drop_duplicates(subset=['Ticker', 'FormationYear']).copy()

n_firms      = df['Ticker'].nunique()
n_firm_years = len(df)
n_years      = df['FormationYear'].nunique()
year_min     = df['FormationYear'].min()
year_max     = df['FormationYear'].max()
avg_per_year = n_firm_years / n_years

summary = pd.DataFrame({
    'Metric': ['Unique firms', 'Firm-years', 'Formation years', 'Period', 'Avg firms / year'],
    'Value':  [n_firms, n_firm_years, n_years, f"{year_min}–{year_max}", f"{avg_per_year:.1f}"]
})
display(summary.style.hide(axis='index').set_properties(**{'text-align': 'left'}))

# ── Plots ─────────────────────────────────────────────────────────────────────
per_year      = df.groupby('FormationYear')['Ticker'].nunique().reset_index()
per_exchange  = df.groupby('Exchange')['Ticker'].nunique().sort_values(ascending=False)
sector_counts = df.groupby(['FormationYear', 'Sector'])['Ticker'].count().unstack(fill_value=0)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Firms per year ────────────────────────────────────────────────────────────
axes[0].bar(per_year['FormationYear'], per_year['Ticker'],
            color=PALETTE['blue'], edgecolor='white', linewidth=0.8)
style_axis(axes[0], title='Firms per formation year',
           xlabel='Formation year', ylabel='Number of firms')

# ── Firms per exchange ────────────────────────────────────────────────────────
axes[1].bar(per_exchange.index, per_exchange.values,
            color=PALETTE['orange'], edgecolor='white', linewidth=0.8)
style_axis(axes[1], title='Unique firms per exchange',
           xlabel='Exchange', ylabel='Number of firms')
axes[1].set_xticklabels(per_exchange.index, rotation=45, ha='right')

# ── Sector composition per year ───────────────────────────────────────────────
sector_counts.plot(kind='bar', stacked=True, ax=axes[2],
                   colormap='tab20', legend=True, width=0.85)
style_axis(axes[2], title='Sector composition per formation year',
           xlabel='Formation year', ylabel='Number of firms')
axes[2].legend(fontsize=7, bbox_to_anchor=(1, 1), title='Sector')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 2. Model Diagnostics

This section examines the behaviour of the Hierarchical Bayesian (HB) model.
We inspect how the model shrinks raw signals toward industry priors, the distribution
of shrinkage weights (λ), accounting uncertainty (σ) across sectors and exchanges,
and how year-level hyperparameters evolve over time.

In [ ]:
# ── Shrinkage: observed vs posterior ─────────────────────────────────────────
sample = firm_year.dropna(subset=['theta_obs', 'theta_post_mean', 'sigma_acc']).copy()
sample = sample[
    sample['theta_obs'].between(
        sample['theta_obs'].quantile(0.02),
        sample['theta_obs'].quantile(0.98)
    )
]

fig, ax = plt.subplots(figsize=(9, 7))

sc = ax.scatter(
    sample['theta_obs'],
    sample['theta_post_mean'],
    c=sample['sigma_acc'],
    cmap='Blues',
    alpha=0.25,
    s=10,
    rasterized=True,
    vmin=sample['sigma_acc'].quantile(0.05),
    vmax=sample['sigma_acc'].quantile(0.95),
)

lims = [
    sample[['theta_obs', 'theta_post_mean']].min().min(),
    sample[['theta_obs', 'theta_post_mean']].max().max(),
]
ax.plot(lims, lims, color='black', linewidth=1, linestyle='--', label='45° — no shrinkage')

plt.colorbar(sc, ax=ax, label='σ_acc (accrual noise)')
style_axis(ax,
           title='Shrinkage: observed vs posterior quality signal',
           xlabel='θ_obs (raw signal)',
           ylabel='θ_post_mean (posterior)')
ax.legend(fontsize=9)
ax.text(0.02, 0.98,
        'Above 45°: shrunk upward\nBelow 45°: shrunk downward\nDarker = higher σ',
        transform=ax.transAxes, va='top', fontsize=9, color=PALETTE['slate'])

plt.tight_layout()
plt.show()

### 2.2 Shrinkage Weights (λ)

The shrinkage weight λ_i indicates how much weight the model places on a firm's
own signal versus the industry prior. A value close to 1 means the firm's own
data dominates; a value close to 0 means the firm is fully pooled toward its
industry.

In [ ]:
# ── Lambda distribution ───────────────────────────────────────────────────────
lam = firm_year['lambda_i'].dropna()

fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(lam, bins=60, color=PALETTE['blue'], edgecolor='white',
        linewidth=0.5, alpha=0.85)

median_lam = lam.median()
ax.axvline(median_lam, color=PALETTE['red'], linewidth=1.5,
           linestyle='--', label=f'Median: {median_lam:.3f}')

style_axis(ax,
           title='Distribution of shrinkage weights (λ)',
           xlabel='λ_i  (1 = full weight on own signal, 0 = fully pooled toward industry prior)',
           ylabel='Frequency')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

# ── Descriptive stats ─────────────────────────────────────────────────────────
desc = lam.describe().round(4).reset_index()
desc.columns = ['Statistic', 'λ_i']
display(desc.style.hide(axis='index'))

### 2.3 Accounting Uncertainty (σ) by Sector and Exchange

The posterior estimate σ_i captures firm-level accounting signal noise. Higher σ
indicates that a firm's accruals are harder to explain from cash flows and structural
controls, making the quality signal less precise.

In [ ]:
# ── Sigma by sector and exchange ──────────────────────────────────────────────
df_diag = firm_year.dropna(subset=['sigma_acc', 'Sector', 'Exchange']).copy()

sector_sigma   = df_diag.groupby('Sector')['sigma_acc'].median().sort_values(ascending=False)
exchange_sigma = df_diag.groupby('Exchange')['sigma_acc'].median().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── By sector ─────────────────────────────────────────────────────────────────
axes[0].bar(sector_sigma.index, sector_sigma.values,
            color=PALETTE['teal'], edgecolor='white', linewidth=0.8)
style_axis(axes[0],
           title='Median σ_acc by sector',
           xlabel='Sector',
           ylabel='Median σ_acc')
axes[0].set_xticklabels(sector_sigma.index, rotation=45, ha='right')
axes[0].axhline(df_diag['sigma_acc'].median(), color=PALETTE['red'],
                linewidth=1.2, linestyle='--', label=f'Overall median: {df_diag["sigma_acc"].median():.3f}')
axes[0].legend(fontsize=9)

# ── By exchange ───────────────────────────────────────────────────────────────
axes[1].bar(exchange_sigma.index, exchange_sigma.values,
            color=PALETTE['orange'], edgecolor='white', linewidth=0.8)
style_axis(axes[1],
           title='Median σ_acc by exchange',
           xlabel='Exchange',
           ylabel='Median σ_acc')
axes[1].set_xticklabels(exchange_sigma.index, rotation=45, ha='right')
axes[1].axhline(df_diag['sigma_acc'].median(), color=PALETTE['red'],
                linewidth=1.2, linestyle='--', label=f'Overall median: {df_diag["sigma_acc"].median():.3f}')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

### 2.4 Year-Level Hyperparameters

The HB model estimates year-level hyperparameters that capture market-wide variation
in quality. μ_t is the cross-sectional mean quality level, τ²_t is the between-firm
variance, and var_obs_t is the observed signal variance. Structural breaks around
crisis years are highlighted.

In [ ]:
# ── Year-level hyperparameters ────────────────────────────────────────────────
year_summary = pd.read_csv(LATENT_DIR / 'latent_prof_year_summary.csv')

CRISIS_YEARS = {2020: 'COVID-19', 2008: 'GFC'}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col, label in zip(
    axes,
    ['mu_t',    'tau2_t',                   'var_obs_t'],
    ['μ_t (market quality mean)',
     'τ²_t (between-firm variance)',
     'var_obs_t (observed variance)'],
):
    ax.plot(year_summary['FormationYear'], year_summary[col],
            marker='o', markersize=4, color=PALETTE['blue'],
            linewidth=2, label=label)

    for crisis_year, crisis_label in CRISIS_YEARS.items():
        if crisis_year in year_summary['FormationYear'].values:
            ax.axvline(crisis_year, color=PALETTE['red'],
                       linewidth=1, linestyle=':',
                       label=crisis_label)

    style_axis(ax, title=label,
               xlabel='Formation year',
               ylabel=label.split(' ')[0])

    # Override formatter to show decimals
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1f}'))
    ax.legend(fontsize=8)

plt.suptitle('Year-level hyperparameters over time', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## 3. Method Comparison

This section examines how the four sorting signals relate to each other.
We compute rank correlations (Spearman) across all firm-years to assess
whether the Bayesian methods produce materially different rankings than
the raw signal.

In [ ]:
# ── Rank correlations across methods ─────────────────────────────────────────
available_signals = {k: v for k, v in SIGNAL_COL.items() if v in firm_year.columns}
available_labels  = {k: LABELS[k] for k in available_signals.keys()}

df_corr = firm_year.dropna(subset=list(available_signals.values())).copy()

corr_matrix = df_corr[list(available_signals.values())].corr(method='spearman')
corr_matrix.index   = list(available_labels.values())
corr_matrix.columns = list(available_labels.values())

fig, ax = plt.subplots(figsize=(8, 6))

im = ax.imshow(corr_matrix.values, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='Spearman rank correlation')

for i in range(len(corr_matrix)):
    for j in range(len(corr_matrix.columns)):
        ax.text(j, i, f'{corr_matrix.values[i, j]:.3f}',
                ha='center', va='center', fontsize=11,
                color='black')

ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.index)))
ax.set_xticklabels(corr_matrix.columns, rotation=30, ha='right', fontsize=10)
ax.set_yticklabels(corr_matrix.index, fontsize=10)

style_axis(ax, title='Spearman rank correlations between sorting signals')

plt.tight_layout()
plt.show()

## 4. Return Analysis

This section analyses portfolio returns under flexible sorting and filtering.
Use the controls below to adjust weighting scheme, industry neutralisation,
size filter, exchange filter, equity ratio filter, and long/short portfolio
composition. Returns are computed dynamically based on the selected parameters.

In [ ]:
w_show_spread = widgets.Checkbox(value=True, description='Spread', layout=widgets.Layout(width='160px'))
w_show_long   = widgets.Checkbox(value=True, description='Long',   layout=widgets.Layout(width='140px'))
w_show_short  = widgets.Checkbox(value=True, description='Short',  layout=widgets.Layout(width='140px'))

w_weight = widgets.Dropdown(
    options=['VW', 'EW', 'SW'],
    value='VW',
    description='Weighting:',
    layout=widgets.Layout(width='160px')
)
w_mode = widgets.Dropdown(
    options=[('Standard', False), ('Industry neutral', True)],
    value=False,
    description='Mode:',
    layout=widgets.Layout(width='200px')
)
w_size = widgets.Dropdown(
    options=['All', 'Large Cap', 'Small Cap'],
    value='All',
    description='Size:',
    layout=widgets.Layout(width='170px')
)
w_exchange = widgets.Dropdown(
    options=['All'] + sorted(holdings_all['Exchange'].dropna().unique().tolist()),
    value='All',
    description='Exchange:',
    layout=widgets.Layout(width='200px')
)
w_ek = widgets.Dropdown(
    options=list(EK_FILTERS.keys()),
    value='No filter',
    description='EK filter:',
    layout=widgets.Layout(width='190px')
)
w_long = widgets.Dropdown(
    options=[label for label, _ in PORTFOLIO_CHOICES],
    value='Q5',
    description='Long:',
    layout=widgets.Layout(width='180px')
)
w_short = widgets.Dropdown(
    options=[label for label, _ in SHORT_CHOICES],
    value='Q1',
    description='Short:',
    layout=widgets.Layout(width='180px')
)
w_factor = widgets.Dropdown(
    options=list(FACTOR_MODELS.keys()),
    value='FF5+MOM',
    description='Factor model:',
    layout=widgets.Layout(width='200px')
)

row1     = widgets.HBox([w_weight, w_mode, w_size, w_exchange, w_ek])
row2     = widgets.HBox([w_long, w_short, w_factor, w_show_spread, w_show_long, w_show_short])
controls = widgets.VBox([row1, row2])

output = widgets.Output()

print("Widgets ready.")

In [ ]:
# ── Get filtered holdings ─────────────────────────────────────────────────────
def get_holdings(weight, mode, size, exchange, ek_label):
    """
    Fetches pre-computed holdings for the given combination.
    If exchange != 'All', applies filter on-the-fly.
    """
    df = PRECOMPUTED[(weight, mode, size, ek_label)].copy()
    if exchange != 'All':
        df = apply_exchange_filter(df, exchange)
    return df


# ── Signal-weighted return ────────────────────────────────────────────────────
def _signal_weighted_return(df, quintiles, method, signal_col):
    sub = df[(df['Portfolio'].isin(quintiles)) & (df['Method'] == method)].copy()
    if sub.empty or signal_col not in sub.columns:
        return pd.Series(dtype=float)
    sub = sub[sub[signal_col].notna() & (sub[signal_col] > 0)]
    monthly = (
        sub.groupby('Date')
        .apply(lambda g: (g['Return'] * g[signal_col]).sum() / g[signal_col].sum())
    )
    return monthly


def _month_end_series(s):
    s = s.copy().dropna()
    s.index = pd.to_datetime(s.index).to_period('M').to_timestamp('M')
    return s.groupby(level=0).last().sort_index()


def _align_excess_return(s, rf_series, subtract_rf):
    s = _month_end_series(s)
    if not subtract_rf:
        return s

    rf = _month_end_series(rf_series)
    common = s.index.intersection(rf.index)
    return s.loc[common] - rf.loc[common]


# ── Compute returns for all methods ──────────────────────────────────────────
def compute_all_returns(holdings_df, weight, long_quintiles, short_quintiles):
    """
    Returns dict with keys = method labels, values = dict with
    'long', 'short', 'spread' as monthly return Series indexed by Date.
    """
    results = {}

    for method, signal_col in SIGNAL_COL.items():
        label = LABELS[method]
        sub   = holdings_df[holdings_df['Method'] == method].copy()

        if weight == 'VW':
            def port_ret(q):
                s = sub[sub['Portfolio'].isin(q)]
                return (
                    s.groupby('Date')
                    .apply(lambda g: (g['Return'] * g['LagMarketCap']).sum() / g['LagMarketCap'].sum())
                )
        elif weight == 'EW':
            def port_ret(q):
                s = sub[sub['Portfolio'].isin(q)]
                return s.groupby('Date')['Return'].mean()
        else:  # SW
            def port_ret(q):
                return _signal_weighted_return(sub, q, method, signal_col)

        long_ret  = port_ret(long_quintiles)
        short_ret = port_ret(short_quintiles) if short_quintiles else None

        if short_ret is not None:
            common = long_ret.index.intersection(short_ret.index)
            spread = long_ret.loc[common] - short_ret.loc[common]
        else:
            spread = None

        results[label] = {
            'long':   long_ret,
            'short':  short_ret,
            'spread': spread,
        }

    return results


# ── Summary statistics ────────────────────────────────────────────────────────
def compute_summary(results, rf_series, factor_model):
    rows = []
    raw_method = 'Method1_Raw'
    raw_label  = LABELS[raw_method]

    selected_returns = {}
    uses_spread = {}

    for method, label in LABELS.items():
        if label not in results:
            continue
        r = results[label]
        use_spread = r['spread'] is not None
        s = r['spread'] if use_spread else r['long']
        if s is None:
            continue
        s = _month_end_series(s)
        s = s[s.index.year >= START_YEAR].dropna()
        if len(s) >= 12:
            selected_returns[method] = s
            uses_spread[method] = use_spread

    all_spreads = bool(selected_returns) and all(uses_spread.values())
    alpha_diff_lookup = {}

    if raw_method in selected_returns and len(selected_returns) > 1:
        try:
            rf_for_alpha = pd.Series(0.0, index=factors.index, name='RF') if all_spreads else rf_series
            diff_df = stacked_alpha_differences(
                ls_returns=selected_returns,
                factors=factors,
                rf=rf_for_alpha,
                model=INTERNAL_FACTOR_MODELS[factor_model],
                lags=12,
                base_method=raw_method,
            )
            alpha_diff_lookup = diff_df.set_index('Comparison').to_dict('index')
        except Exception as exc:
            print(f"Alpha-difference helper failed: {exc}")

    for method, label in LABELS.items():
        if method not in selected_returns:
            continue

        s = selected_returns[method]
        excess_s = _align_excess_return(s, rf_series, subtract_rf=not uses_spread[method])
        if excess_s.empty:
            continue

        ann_ret = ((1 + s.mean()) ** 12 - 1) * 100
        ann_vol = s.std() * np.sqrt(12) * 100
        sharpe  = (excess_s.mean() * 12 * 100) / ann_vol if ann_vol > 0 else np.nan

        cum    = (1 + s).cumprod()
        peak   = cum.cummax()
        max_dd = ((cum - peak) / peak).min() * 100

        if method == raw_method:
            alpha_vs_raw = t_stat = p_val = None
        else:
            comp = f"{method} vs {raw_method}"
            diff = alpha_diff_lookup.get(comp)
            if diff is None:
                alpha_vs_raw = t_stat = p_val = None
            else:
                alpha_vs_raw = diff['alpha_difference'] * 12 * 100
                t_stat = diff['t_stat']
                p_val = diff['p_value']

        rows.append({
            'Method':        label,
            'Ann. return':   ann_ret,
            'Ann. vol':      ann_vol,
            'Sharpe':        sharpe,
            'Max DD':        max_dd,
            'α vs Raw (%)':  alpha_vs_raw,
            't-stat':        t_stat,
            'p-value':       p_val,
        })

    return pd.DataFrame(rows)


# ── Factor regression ─────────────────────────────────────────────────────────
def run_factor_regression(ret_series, factor_cols, subtract_rf=True):
    s = _month_end_series(ret_series)

    f = factors[factor_cols + ['RF']].copy()
    f.index = f.index.to_period('M').to_timestamp('M')

    df = pd.DataFrame({'ret': s}).join(f).dropna()
    if len(df) < 24:
        return None

    df['excess'] = df['ret'] - df['RF'] if subtract_rf else df['ret']
    X     = sm.add_constant(df[factor_cols])
    model = sm.OLS(df['excess'], X).fit(cov_type='HAC', cov_kwds={'maxlags': 12})
    return {
        'alpha':   model.params['const'] * 12 * 100,
        't_stat':  model.tvalues['const'],
        'p_value': model.pvalues['const'],
        'betas':   {f: model.params[f] for f in factor_cols},
        'n_obs':   len(df),
    }


print("Helper functions ready.")


In [ ]:
# ── Plot A: Cumulative returns ─────────────────────────────────────────────────
def plot_cumulative(results, show_spread, show_long, show_short, ax):
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')

    for method, label in LABELS.items():
        color = METHOD_COLORS.get(method, PALETTE['slate'])
        if label not in results:
            continue
        r = results[label]

        if show_spread and r['spread'] is not None:
            s = r['spread'][r['spread'].index.year >= START_YEAR].dropna()
            cum = (1 + s).cumprod() - 1
            ax.plot(cum.index, cum * 100, color=color, linewidth=2.5,
                    linestyle='-', label=f'{label} (spread)')

        if show_long and r['long'] is not None:
            s = r['long'][r['long'].index.year >= START_YEAR].dropna()
            cum = (1 + s).cumprod() - 1
            ax.plot(cum.index, cum * 100, color=color, linewidth=1.5,
                    linestyle='--', alpha=0.7, label=f'{label} (long)')

        if show_short and r['short'] is not None:
            s = r['short'][r['short'].index.year >= START_YEAR].dropna()
            cum = (1 + s).cumprod() - 1
            ax.plot(cum.index, cum * 100, color=color, linewidth=1.5,
                    linestyle=':', alpha=0.7, label=f'{label} (short)')

    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    style_axis(ax, title='Cumulative returns', xlabel='Date', ylabel='Cumulative return (%)')
    ax.legend(fontsize=8, ncol=2)


# ── Plot B: Alpha heatmap ──────────────────────────────────────────────────────
def plot_alpha_heatmap(results, factor_model, long_label, short_label, ax):
    method_labels = list(LABELS.values())
    model_names   = list(FACTOR_MODELS.keys())

    alphas = np.full((len(model_names), len(method_labels)), np.nan)
    tstats = np.full((len(model_names), len(method_labels)), np.nan)

    for j, label in enumerate(method_labels):
        if label not in results:
            continue
        r = results[label]
        use_spread = r['spread'] is not None
        s = r['spread'] if use_spread else r['long']
        if s is None:
            continue
        s = s[s.index.year >= START_YEAR].dropna()

        for i, (model_name, fcols) in enumerate(FACTOR_MODELS.items()):
            res = run_factor_regression(s, fcols, subtract_rf=not use_spread)
            if res:
                alphas[i, j] = res['alpha']
                tstats[i, j] = res['t_stat']

    im = ax.imshow(alphas, cmap='RdYlGn', aspect='auto', vmin=-10, vmax=10)
    plt.colorbar(im, ax=ax, label='Alpha (annualised %)')

    ax.set_xticks(range(len(method_labels)))
    ax.set_yticks(range(len(model_names)))
    ax.set_xticklabels(method_labels, rotation=20, ha='right', fontsize=9)
    ax.set_yticklabels(model_names, fontsize=9)

    for i in range(len(model_names)):
        for j in range(len(method_labels)):
            if np.isnan(alphas[i, j]):
                continue
            stars = (
                '***' if abs(tstats[i, j]) > 3.09 else
                '**'  if abs(tstats[i, j]) > 2.33 else
                '*'   if abs(tstats[i, j]) > 1.65 else ''
            )
            ax.text(j, i, f'{alphas[i, j]:.1f}%{stars}\n({tstats[i, j]:.2f})',
                    ha='center', va='center', fontsize=8, color='black')

    short_str = f'−{short_label}' if short_label != 'None' else ''
    style_axis(ax, title=f'Annualised alpha ({long_label}{short_str})')


# ── Plot C1: Alpha per quintile ───────────────────────────────────────────────
def plot_alpha_by_quintile(holdings_df, factor_model, weight, ax):
    factor_cols = FACTOR_MODELS[factor_model]
    x     = np.arange(len(QUINTILES))
    width = 0.2

    for i, (method, label) in enumerate(LABELS.items()):
        if method not in SIGNAL_COL:
            continue
        color  = METHOD_COLORS.get(method, PALETTE['slate'])
        alphas = []

        for q in QUINTILES:
            sub = holdings_df[
                (holdings_df['Method'] == method) &
                (holdings_df['Portfolio'] == q)
            ]
            if weight == 'VW':
                s = (
                    sub.groupby('Date')
                    .apply(lambda g: (g['Return'] * g['LagMarketCap']).sum() / g['LagMarketCap'].sum())
                )
            elif weight == 'EW':
                s = sub.groupby('Date')['Return'].mean()
            else:
                s = _signal_weighted_return(sub, [q], method, SIGNAL_COL[method])

            s   = s[s.index.year >= START_YEAR].dropna()
            res = run_factor_regression(s, factor_cols, subtract_rf=True)
            alphas.append(res['alpha'] if res else np.nan)

        ax.bar(x + i * width, alphas, width,
               label=label, color=color,
               edgecolor='white', linewidth=0.8, alpha=0.85)

    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xticks(x + width * (len(LABELS) - 1) / 2)
    ax.set_xticklabels(QUINTILES)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1f}%'))
    style_axis(ax, title=f'{factor_model} alpha by quintile',
               xlabel='Quintile', ylabel='Annualised alpha (%)')
    ax.legend(fontsize=8)


# ── Plot C2: Factor loadings ───────────────────────────────────────────────────
def plot_factor_loadings(results, factor_model, long_label, short_label, ax):
    factor_cols = FACTOR_MODELS[factor_model]
    x     = np.arange(len(factor_cols))
    width = 0.2

    for i, (method, label) in enumerate(LABELS.items()):
        color = METHOD_COLORS.get(method, PALETTE['slate'])
        if label not in results:
            continue
        r = results[label]
        use_spread = r['spread'] is not None
        s = r['spread'] if use_spread else r['long']
        if s is None:
            continue
        s   = s[s.index.year >= START_YEAR].dropna()
        res = run_factor_regression(s, factor_cols, subtract_rf=not use_spread)
        if res is None:
            continue

        betas = [res['betas'].get(f, np.nan) for f in factor_cols]
        ax.bar(x + i * width, betas, width,
               label=label, color=color,
               edgecolor='white', linewidth=0.8, alpha=0.85)

    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xticks(x + width * (len(LABELS) - 1) / 2)
    ax.set_xticklabels(factor_cols)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.2f}'))
    short_str = f'−{short_label}' if short_label != 'None' else ''
    style_axis(ax, title=f'Factor loadings ({long_label}{short_str}) {factor_model}',
               xlabel='Factor', ylabel='Beta')
    ax.legend(fontsize=8)


print("Plot functions ready.")


In [ ]:
# ── Main update function ───────────────────────────────────────────────────────
def update(change=None):
    with output:
        clear_output(wait=True)

        # ── Get current widget values ─────────────────────────────────────────
        weight      = w_weight.value
        mode        = w_mode.value
        size        = w_size.value
        exchange    = w_exchange.value
        ek_label    = w_ek.value
        long_label  = w_long.value
        short_label = w_short.value
        factor_model = w_factor.value
        show_spread = w_show_spread.value
        show_long   = w_show_long.value
        show_short  = w_show_short.value

        long_quintiles  = PORTFOLIO_LOOKUP[long_label]
        short_quintiles = SHORT_LOOKUP[short_label]

        # ── Get holdings ──────────────────────────────────────────────────────
        holdings_df = get_holdings(weight, mode, size, exchange, ek_label)

        # ── Compute returns ───────────────────────────────────────────────────
        results = compute_all_returns(holdings_df, weight, long_quintiles, short_quintiles)

        # ── Figure layout ─────────────────────────────────────────────────────
        fig = plt.figure(figsize=(18, 22))
        gs  = fig.add_gridspec(4, 2, hspace=0.45, wspace=0.35)

        ax_cum      = fig.add_subplot(gs[0, :])      # full width
        ax_heatmap  = fig.add_subplot(gs[1, :])      # full width
        ax_quintile = fig.add_subplot(gs[2, :])      # full width
        ax_loadings = fig.add_subplot(gs[3, :])      # full width

        # ── Title ─────────────────────────────────────────────────────────────
        short_str = f'−{short_label}' if short_label != 'None' else ''
        fig.suptitle(
            f'{long_label}{short_str}  —  {weight}  —  {"Industry neutral" if mode else "Standard"}  —  {size}  —  {exchange}  —  {ek_label}',
            fontsize=13, y=0.98
        )

        # ── Plot A ────────────────────────────────────────────────────────────
        plot_cumulative(results, show_spread, show_long, show_short, ax_cum)

        # ── Plot B ────────────────────────────────────────────────────────────
        plot_alpha_heatmap(results, factor_model, long_label, short_label, ax_heatmap)

        # ── Plot C1 ───────────────────────────────────────────────────────────
        plot_alpha_by_quintile(holdings_df, factor_model, weight, ax_quintile)

        # ── Plot C2 ───────────────────────────────────────────────────────────
        plot_factor_loadings(results, factor_model, long_label, short_label, ax_loadings)

        plt.show()

        # ── Summary table ─────────────────────────────────────────────────────
        rf = factors['RF'] if 'RF' in factors.columns else pd.Series(0, index=factors.index)
        summary_df = compute_summary(results, rf, factor_model)

        def fmt_summary(df):
            styled = df.style.hide(axis='index')
            styled = styled.format({
                'Ann. return':  '{:.1f}%',
                'Ann. vol':     '{:.1f}%',
                'Sharpe':       '{:.2f}',
                'Max DD':       '{:.1f}%',
                'α vs Raw (%)': lambda x: f'{x:+.1f}%' if x is not None else '—',
                't-stat':       lambda x: f'{x:.2f}' if x is not None else '—',
                'p-value':      lambda x: f'{x:.3f}' if x is not None else '—',
            })
            return styled

        display(fmt_summary(summary_df))


# ── Observe all widgets ────────────────────────────────────────────────────────
for w in [w_weight, w_mode, w_size, w_exchange, w_ek,
          w_long, w_short, w_factor,
          w_show_spread, w_show_long, w_show_short]:
    w.observe(update, names='value')

# ── Display ────────────────────────────────────────────────────────────────────
display(controls)
display(output)
update()


In [ ]:
print("Unike tickers i Q5 per FormationYear (Method1_Raw) — holdings_all:")
print(
    holdings_all[
        (holdings_all['Method'] == 'Method1_Raw') &
        (holdings_all['Portfolio'] == 'Q5')
    ]
    .groupby('FormationYear')['Ticker']
    .nunique()
    .sort_index()
)